In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import cv2
import os

# path = "/content/drive/My Drive"
# os.chdir(path)

def load_files(path):
  x_train = []
  y_train = []
  for file_name in sorted(os.listdir(path)):
    if '.npy' in file_name:
      y_train.append(np.load(path + file_name))
    else:
      x_train.append(cv2.imread(path + file_name))

  x_train,y_train = np.array(x_train),np.array(y_train,dtype='uint8') # float32
  x_test,y_test = x_train [-30:], y_train[-30:]
  x_train,y_train = x_train [:-30], y_train[:-30]
  return x_train,y_train,x_test,y_test

path = "/content/drive/MyDrive/vaihingen/vaihingen_train/"
x_train,y_train,x_test,y_test = load_files(path)

print("train images",len(y_train))
print("test images",len(y_test))
print("classes ",np.max(y_train)+1)

train images 127
test images 30
classes  6


In [ ]:
input_shape = [512,512,3]

In [ ]:
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import numpy as np
import os
from tensorflow.keras.layers import Conv2D,BatchNormalization,Activation,MaxPool2D,Conv2DTranspose,Concatenate,Input
from tensorflow.keras.models import Model
import tensorflow as tf
from sklearn.model_selection import train_test_split

def conv_block(inputs,filterCount):
  x = Conv2D(filterCount,3,padding ="same")(inputs)
  x = BatchNormalization()(x)
  x = Activation("relu")(x)

  x = Conv2D(filterCount,3,padding ="same")(x)
  x = BatchNormalization()(x)
  x = Activation("relu")(x)

  return x

def decoder_block(inputs,skip_features,filter_count):
  x = Conv2DTranspose(filter_count,(2,2),strides=2,padding ="same")(inputs)
  x = Concatenate()([x,skip_features])
  x = conv_block(x,filter_count)
  return x

In [ ]:
# mixed_float16

from tensorflow.keras.applications import EfficientNetB0
tf.keras.mixed_precision.set_global_policy('mixed_float16')

inputs = Input(input_shape)
convnextlarge = EfficientNetB0(include_top=False,weights='imagenet',input_tensor=inputs)

16705208/16705208 [==============================] - 0s 0us/step


In [ ]:
s1 = convnextlarge.get_layer("rescaling_1").output
s2 = convnextlarge.get_layer("block2a_expand_activation").output
s3 = convnextlarge.get_layer("block3a_expand_activation").output
s4 = convnextlarge.get_layer("block4a_expand_activation").output
center = convnextlarge.get_layer("block6a_expand_activation").output

d1 = decoder_block(center,s4,256)
d2 = decoder_block(d1,s3,128)
d3 = decoder_block(d2,s2,64)
d4 = decoder_block(d3,s1,32)
# output
conv1 = Conv2D(32,3,padding="same")(d4)
conv2 = Conv2D(16,3,padding="same")(conv1)
outputs = Conv2D(6,1,padding="same",activation="softmax",dtype=tf.float32)(conv2)
model = Model(inputs,outputs,name="EfficientNetB0_U_Net")

In [ ]:
convnextlarge.summary()

Model: "efficientnetb0"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 512, 512, 3)]        0         []                            
                                                                                                  
 rescaling (Rescaling)       (None, 512, 512, 3)          0         ['input_1[0][0]']             
                                                                                                  
 normalization (Normalizati  (None, 512, 512, 3)          7         ['rescaling[0][0]']           
 on)                                                                                              
                                                                                                  
 rescaling_1 (Rescaling)     (None, 512, 512, 3)          0         ['normalization[0

In [ ]:
from keras.src import metrics
from scipy.optimize import optimize
from keras.src.utils.sidecar_evaluator import optimizer
batch_size = 1
epochs = 10

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss="sparse_categorical_crossentropy",optimizer= optimizer,metrics=["accuracy"])

In [ ]:
import time

start_time = time.time()

for i in range(epochs):
  print('Epocs:',i+1)
  model.fit(x_train,y_train,batch_size=batch_size,epochs=1)
  print('Train accuracy:')
  model.evaluate(x_train,y_train,batch_size=1)
  print('Test accuracy:')
  model.evaluate(x_test,y_test,batch_size=1)

print("Время выполнения ", time.time() - start_time)

Epocs: 1
127/127 [==============================] - 71s 78ms/step - loss: 0.9927 - accuracy: 0.6310
Train accuracy:
127/127 [==============================] - 5s 21ms/step - loss: 1.3201 - accuracy: 0.6646
Test accuracy:
30/30 [==============================] - 1s 19ms/step - loss: 1.3507 - accuracy: 0.6977
Epocs: 2
127/127 [==============================] - 9s 73ms/step - loss: 0.7908 - accuracy: 0.7106
Train accuracy:
127/127 [==============================] - 3s 20ms/step - loss: 1.2573 - accuracy: 0.7097
Test accuracy:
30/30 [==============================] - 1s 20ms/step - loss: 1.4130 - accuracy: 0.6772
Epocs: 3
127/127 [==============================] - 9s 73ms/step - loss: 0.6792 - accuracy: 0.7617
Train accuracy:
127/127 [==============================] - 3s 21ms/step - loss: 0.8753 - accuracy: 0.7238
Test accuracy:
30/30 [==============================] - 1s 21ms/step - loss: 0.8041 - accuracy: 0.7485
Epocs: 4
127/127 [==============================] - 10s 75ms/step - loss: 0

In [ ]:
mIoU = tf.keras.metrics.MeanIoU(num_classes=6)
ot_model = model.predict(x_train,batch_size=1)
mIoU.update_state(np.argmax(ot_model,axis=-1),y_train)
print('Train mIoU:',mIoU.result().numpy())

mIoU = tf.keras.metrics.MeanIoU(num_classes=6)
ot_model = model.predict(x_test,batch_size=1)
mIoU.update_state(np.argmax(ot_model,axis=-1),y_test)
print('Test mIoU:',mIoU.result().numpy())

127/127 [==============================] - 5s 19ms/step
Train mIoU: 0.63622195
30/30 [==============================] - 0s 16ms/step
Test mIoU: 0.5133969
